# M5 Forecasting — California LightGBM Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os

CACHE_FILE = "df_CA_cache.parquet"

if os.path.exists(CACHE_FILE):
    df_CA = pd.read_parquet(CACHE_FILE)
    _from_cache = True
    print(f"Loaded from cache: {df_CA.shape}")
    print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
else:
    _from_cache = False
    print("No cache found — run sections 1–3 below.")

Loaded from cache: (23330948, 23)
Memory: 2010.4 MB


## 1. Load Sales Data & Extract CA Subset

In [ ]:
if not _from_cache:
    df_sales = pd.read_csv("sales_train_validation.csv")
    df_sales_CA = df_sales[df_sales["state_id"] == "CA"].copy()
    del df_sales

    id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    sales_cols = [col for col in df_sales_CA.columns if col.startswith("d_")]

    df_CA_long = df_sales_CA.melt(
        id_vars=id_cols, value_vars=sales_cols, var_name="d", value_name="sales"
    )
    del df_sales_CA
    df_CA_long["d_num"] = df_CA_long["d"].str.extract(r"d_(\d+)").astype("uint16")
    df_CA_long = df_CA_long.astype({
        "item_id": "category", "dept_id": "category", "cat_id": "category",
        "store_id": "category", "state_id": "category", "d": "category", "sales": "uint16",
    })
    print("CA long shape:", df_CA_long.shape)

## 2. Load Calendar & Merge

In [ ]:
if not _from_cache:
    df_calendar = pd.read_csv("calendar.csv")
    for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
        df_calendar[col] = df_calendar[col].fillna("No_Event")
    df_calendar = df_calendar.astype({
        "wm_yr_wk": "uint16", "wday": "uint8", "month": "uint8", "year": "uint16",
        "weekday": "category", "event_name_1": "category", "event_type_1": "category",
        "event_name_2": "category", "event_type_2": "category",
        "snap_CA": "bool", "snap_TX": "bool", "snap_WI": "bool",
    })
    df_calendar["date"] = pd.to_datetime(df_calendar["date"]).astype("datetime64[ms]")
    df_CA_merged = df_CA_long.merge(df_calendar, on="d", how="left")
    del df_calendar
    print("Merged shape:", df_CA_merged.shape)

## 3. Load Sell Prices & Merge

In [ ]:
if not _from_cache:
    df_sell_prices = pd.read_csv("sell_prices.csv")
    df_sell_prices_CA = df_sell_prices[df_sell_prices["store_id"].str.startswith("CA")].copy()
    del df_sell_prices
    df_sell_prices_CA = df_sell_prices_CA.astype({
        "store_id": "category", "item_id": "category",
        "wm_yr_wk": "uint16", "sell_price": "float32",
    })
    df_CA = df_CA_merged.merge(df_sell_prices_CA, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    del df_CA_merged, df_sell_prices_CA
    df_CA.to_parquet(CACHE_FILE, index=False)
    print(f"Cache saved → {CACHE_FILE}")

print("df_CA shape:", df_CA.shape)
print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 1a: Sort & Log-Transform Target

In [3]:
print("Sorting by item_id, store_id, date...")
df_CA = df_CA.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)

# Log1p transform — stabilises variance and compresses outliers
# Training target: sales_log; recover predictions with np.expm1()
df_CA["sales_log"] = np.log1p(df_CA["sales"].astype("float32")).astype("float32")

print(f"Shape: {df_CA.shape}")
print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nsales_log sample stats:")
print(df_CA["sales_log"].describe())

Sorting by item_id, store_id, date...
Shape: (23330948, 24)
Memory: 2109.5 MB

sales_log sample stats:
count    2.333095e+07
mean     4.205405e-01
std      6.968054e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.931472e-01
max      6.638568e+00
Name: sales_log, dtype: float64


## Step 1b: Lag Features (log scale)

In [4]:
print("Computing lag_7 and lag_28 on log scale...")
grp = df_CA.groupby(["item_id", "store_id"], observed=True)["sales_log"]
df_CA["lag_7_log"]  = grp.shift(7).astype("float32")
df_CA["lag_28_log"] = grp.shift(28).astype("float32")

print("Computing lag_364 (leap-year aware)...")
prev_year     = df_CA["date"].dt.year - 1
lookback_days = prev_year.map(lambda y: 366 if calendar.isleap(y) else 365)
df_CA["_lag_date"] = df_CA["date"] - pd.to_timedelta(lookback_days, unit="D")

lookup = (
    df_CA[["item_id", "store_id", "date", "sales_log"]]
    .rename(columns={"date": "_lag_date", "sales_log": "lag_364_log"})
)
df_CA = df_CA.merge(lookup, on=["item_id", "store_id", "_lag_date"], how="left")
df_CA["lag_364_log"] = df_CA["lag_364_log"].astype("float32")
df_CA = df_CA.drop(columns=["_lag_date"])

print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nNull counts (expected — first N days have no prior data):")
print(df_CA[["lag_7_log", "lag_28_log", "lag_364_log"]].isnull().sum())

Computing lag_7 and lag_28 on log scale...
Computing lag_364 (leap-year aware)...
Memory: 2389.5 MB

Null counts (expected — first N days have no prior data):
lag_7_log        85372
lag_28_log      341488
lag_364_log    4451540
dtype: int64


## Step 1c: Rolling Features

In [5]:
print("Computing rolling features on log scale (shift(1) prevents leakage)...")
grp_log = df_CA.groupby(["item_id", "store_id"], observed=True)["sales_log"]

df_CA["rolling_mean_7"]  = grp_log.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).mean()
).astype("float32")

df_CA["rolling_mean_28"] = grp_log.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
).astype("float32")

df_CA["rolling_std_7"]   = grp_log.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).std().fillna(0)
).astype("float32")

# rolling_zero_count on raw sales (intermittency signal)
print("Computing rolling_zero_count_7 on raw sales...")
grp_raw = df_CA.groupby(["item_id", "store_id"], observed=True)["sales"]
df_CA["rolling_zero_count_7"] = grp_raw.transform(
    lambda x: (x.shift(1) == 0).rolling(7, min_periods=1).sum()
).astype("float32")

print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nRolling feature null counts:")
print(df_CA[["rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_zero_count_7"]].isnull().sum())

Computing rolling features on log scale (shift(1) prevents leakage)...
Computing rolling_zero_count_7 on raw sales...
Memory: 2762.8 MB

Rolling feature null counts:
rolling_mean_7          12196
rolling_mean_28         12196
rolling_std_7               0
rolling_zero_count_7        0
dtype: int64


## Step 1d: Calendar Features

In [6]:
df_CA["day_of_week"]  = df_CA["date"].dt.dayofweek.astype("int8")        # 0=Mon, 6=Sun
df_CA["month"]        = df_CA["date"].dt.month.astype("int8")
df_CA["week_of_year"] = df_CA["date"].dt.isocalendar().week.astype("int8")
df_CA["is_weekend"]   = (df_CA["date"].dt.dayofweek >= 5).astype("uint8")
df_CA["day_of_month"] = df_CA["date"].dt.day.astype("int8")

print("Calendar features added.")
print(df_CA[["day_of_week", "month", "week_of_year", "is_weekend", "day_of_month"]].dtypes)

Calendar features added.
day_of_week      int8
month            int8
week_of_year     int8
is_weekend      uint8
day_of_month     int8
dtype: object


## Step 1e: Event Interaction Features

In [7]:
# EDA showed FOODS +25-49% lift, HOBBIES/HOUSEHOLD -36-55% suppression
# A single event flag across categories would mislead the model
# Interaction encodes the category-specific effect

df_CA["event_indicator"] = (df_CA["event_type_1"] != "No_Event").astype("uint8")

df_CA["cat_event"] = (
    df_CA["cat_id"].astype(str) + "_" + df_CA["event_type_1"].astype(str)
).astype("category")

print("Event interaction features added.")
print("Unique cat_event values:", df_CA["cat_event"].nunique())
print(df_CA["cat_event"].value_counts().head(10))

Event interaction features added.
Unique cat_event values: 15
cat_event
FOODS_No_Event         10110732
HOUSEHOLD_No_Event      7366692
HOBBIES_No_Event        3975340
FOODS_Religious          298896
FOODS_National           293148
HOUSEHOLD_Religious      217776
HOUSEHOLD_National       213588
FOODS_Cultural           201180
HOUSEHOLD_Cultural       146580
HOBBIES_Religious        117520
Name: count, dtype: int64


## Step 1f: Price Features

In [8]:
# Raw sell_price has weak correlation (|r| < 0.2 from EDA)
# Price *changes* carry the promo signal — nearly 50% of items had >10% markdown

grp_price = df_CA.groupby(["item_id", "store_id"], observed=True)["sell_price"]

# price_change_7: % change vs 7 days ago (captures weekly markdown events)
price_lag7 = grp_price.shift(7).astype("float32")
df_CA["price_change_7"] = (
    (df_CA["sell_price"] - price_lag7) / (price_lag7.fillna(1) + 1e-8)
).astype("float32")

# price_ratio_28: current price relative to 28-day rolling baseline
price_roll28 = grp_price.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
).astype("float32")
df_CA["price_ratio_28"] = (
    df_CA["sell_price"] / (price_roll28.fillna(1) + 1e-8)
).astype("float32")

print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nPrice feature null counts:")
print(df_CA[["price_change_7", "price_ratio_28"]].isnull().sum())

Memory: 3089.4 MB

Price feature null counts:
price_change_7    5198522
price_ratio_28    5113150
dtype: int64


## Step 1g: Drop Null Rows

In [9]:
shape_before = df_CA.shape

# lag_364_log is null for all of 2011 (no prior year)
# lag_7_log, lag_28_log are null for first 7/28 days of each series
# rolling features have min_periods=1 so minimal nulls

LAG_COLS = ["lag_7_log", "lag_28_log", "lag_364_log"]
df_model = df_CA.dropna(subset=LAG_COLS).reset_index(drop=True)

print(f"Before: {shape_before[0]:,} rows")
print(f"After:  {df_model.shape[0]:,} rows")
print(f"Dropped: {shape_before[0] - df_model.shape[0]:,} rows")
print(f"Date range after drop: {df_model['date'].min()} → {df_model['date'].max()}")
print(f"Memory: {df_model.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Before: 23,330,948 rows
After:  18,879,408 rows
Dropped: 4,451,540 rows
Date range after drop: 2012-01-29 00:00:00 → 2016-04-24 00:00:00
Memory: 2503.2 MB


## Step 1h: Define Feature Columns

In [10]:
TARGET = "sales_log"

FEATURE_COLS = [
    # Lag features
    "lag_7_log", "lag_28_log", "lag_364_log",
    # Rolling features
    "rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_zero_count_7",
    # Calendar
    "day_of_week", "month", "week_of_year", "is_weekend", "day_of_month",
    # Event
    "event_indicator", "cat_event",
    # SNAP
    "snap_CA",
    # Price
    "sell_price", "price_change_7", "price_ratio_28",
    # Identifiers (LightGBM uses these as categoricals)
    "store_id", "item_id", "dept_id", "cat_id",
]

CAT_COLS = ["store_id", "item_id", "dept_id", "cat_id", "cat_event"]

# Ensure categorical dtype for LightGBM native categorical support
for col in CAT_COLS:
    df_model[col] = df_model[col].astype("category")

print(f"Target:   {TARGET}")
print(f"Features: {len(FEATURE_COLS)}")
print()
for f in FEATURE_COLS:
    print(f"  {f:<28} {str(df_model[f].dtype)}")
print(f"\ndf_model shape: {df_model.shape}")
print(f"Memory:         {df_model.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Target:   sales_log
Features: 22

  lag_7_log                    float32
  lag_28_log                   float32
  lag_364_log                  float32
  rolling_mean_7               float32
  rolling_mean_28              float32
  rolling_std_7                float32
  rolling_zero_count_7         float32
  day_of_week                  int8
  month                        int8
  week_of_year                 int8
  is_weekend                   uint8
  day_of_month                 int8
  event_indicator              uint8
  cat_event                    category
  snap_CA                      bool
  sell_price                   float32
  price_change_7               float32
  price_ratio_28               float32
  store_id                     category
  item_id                      category
  dept_id                      category
  cat_id                       category

df_model shape: (18879408, 39)
Memory:         2503.2 MB


## Step 2: Train / Validation Split

In [ ]:
# M5 evaluation window = last 28 days of known data (d_1886 → d_1913)
# Val mirrors the competition holdout window in length and recency
# Train: everything before that (2012-01-01 onwards after null drop)

VAL_DAYS = 28
max_date = df_model["date"].max()
val_start = max_date - pd.Timedelta(days=VAL_DAYS - 1)

df_train = df_model[df_model["date"] < val_start].reset_index(drop=True)
df_val   = df_model[df_model["date"] >= val_start].reset_index(drop=True)

print(f"Train: {df_train.shape[0]:>12,} rows  |  {df_train['date'].min().date()} → {df_train['date'].max().date()}")
print(f"Val:   {df_val.shape[0]:>12,} rows  |  {df_val['date'].min().date()} → {df_val['date'].max().date()}")
print(f"\nVal items  × stores: {df_val['item_id'].nunique()} items × {df_val['store_id'].nunique()} stores")
print(f"Train items × stores: {df_train['item_id'].nunique()} items × {df_train['store_id'].nunique()} stores")

### Target Distribution — Train vs Val

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# sales_log distribution
axes[0].hist(df_train["sales_log"], bins=50, color="steelblue", alpha=0.7, label="Train", density=True)
axes[0].hist(df_val["sales_log"],   bins=50, color="tomato",    alpha=0.7, label="Val",   density=True)
axes[0].set_title("sales_log distribution")
axes[0].set_xlabel("log1p(sales)")
axes[0].legend()

# Daily total sales over time with val window shaded
daily = df_model.groupby("date")["sales"].sum()
axes[1].plot(daily.index, daily.values, color="steelblue", linewidth=0.8)
axes[1].axvspan(val_start, max_date, color="tomato", alpha=0.25, label=f"Val ({VAL_DAYS}d)")
axes[1].set_title("Total Daily Sales — Val Window")
axes[1].set_xlabel("Date")
axes[1].legend()

plt.tight_layout()
plt.savefig("ca_train_val_split.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ca_train_val_split.png")

### Prepare X / y Arrays

In [ ]:
X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET]

X_val = df_val[FEATURE_COLS]
y_val = df_val[TARGET]

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}  |  y_val:   {y_val.shape}")
print(f"\nFeature dtypes:")
print(X_train.dtypes.to_string())

## Step 3: Train LightGBM

In [ ]:
import lightgbm as lgb

print("Building LightGBM datasets...")
dtrain = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_COLS,
    free_raw_data=False
)
dval = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=CAT_COLS,
    reference=dtrain,
    free_raw_data=False
)
print("Datasets ready.")

### Hyperparameters

In [ ]:
params = {
    "objective":        "regression",       # predict log1p(sales)
    "metric":           "rmse",
    "verbosity":        -1,
    "boosting_type":    "gbdt",
    "num_leaves":       127,                # tree complexity
    "learning_rate":    0.05,
    "feature_fraction": 0.8,               # feature subsampling per tree
    "bagging_fraction": 0.8,               # row subsampling
    "bagging_freq":     5,
    "min_child_samples": 50,               # min rows per leaf — prevents overfit on rare items
    "lambda_l1":        0.1,
    "lambda_l2":        0.1,
    "seed":             42,
}

print("Params:")
for k, v in params.items():
    print(f"  {k:<22} {v}")

### Train with Early Stopping

In [ ]:
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

print("Training...")
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=callbacks,
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best val RMSE:  {model.best_score['val']['rmse']:.4f}")

### Save Model

In [ ]:
model.save_model("lgb_baseline.txt")
print("Model saved → lgb_baseline.txt")
print(f"  Best iteration: {model.best_iteration}")
print(f"  Val RMSE (log scale): {model.best_score['val']['rmse']:.4f}")
print(f"  Val RMSE (original scale approx): expm1 of above is not directly comparable")